# 01 — Crash Frequency Model Search

This notebook runs a **mixed Negative Binomial crash-frequency model search**
on the Example 16-3 road-segment dataset.

**What you will do:**
1. Load and inspect the bundled dataset
2. Declare structural constraints with `ModelConstraints`
3. Build a structure evaluator (defines the search space)
4. Run a Simulated Annealing search to minimise BIC
5. Inspect the best model structure
6. Re-fit with more simulation draws for accurate standard errors
7. Fit a fully manual specification to compare

**Previous notebook:** [00_quickstart.ipynb](00_quickstart.ipynb)
**Next notebook:** [02_latent_class_fc_validation.ipynb](02_latent_class_fc_validation.ipynb)

In [ ]:
import numpy as np
import pandas as pd

from metacountregressor import (
    ExperimentBuilder,
    ModelConstraints,
    SearchOutputConfig,
    load_example16_3_model_data,
    get_help,
)

# Uncomment any of these to read the full guide for that topic:
# get_help()                   # list all topics
# get_help('crash_frequency')  # end-to-end count model workflow
# get_help('constraints')      # ModelConstraints API
# get_help('roles')            # role code reference
# get_help('metaheuristics')   # algorithm comparison

print('Imports OK')

## 1 — Load data

In [ ]:
df = load_example16_3_model_data()

# The OFFSET column is log(LENGTH * AADT * 365 / 1e8) — already in the loader.
# To build it yourself:
#   exposure = df['LENGTH'] * df['AADT'] * 365 / 1e8
#   df['OFFSET'] = np.log(exposure.clip(lower=1e-9))

print(f'Rows: {len(df)}  |  Columns: {df.shape[1]}')
print()
print('Outcome summary (FREQ = crash count):')
print(df['FREQ'].describe().round(2))
print()
print('Offset summary (log VMT):')
print(df['OFFSET'].describe().round(3))
print()
print('Functional class distribution:')
print(df['FC_LABEL'].value_counts().sort_index())

In [ ]:
df[['ID', 'FREQ', 'AADT', 'LENGTH', 'SPEED', 'URB', 'CURVES', 'AVEPRE', 'OFFSET']].head(6)

## 2 — Select candidate variables and set constraints

`ModelConstraints` controls what roles each variable is **allowed** to take.
It does not force the search to pick a role — it only rules out implausible ones.

**Role codes quick reference:**

| Code | Name | Use when ... |
| --- | --- | --- |
| 0 | Excluded | Variable not needed |
| 1 | Fixed | Homogeneous effect across all sites |
| 2 | Random (ind.) | Heterogeneous effect — independent |
| 3 | Random (corr.) | Heterogeneous effect — correlated with other randoms |
| 4 | Grouped | Shared random effect within a group |
| 5 | Heterogeneity | Explains spread in random-parameter means |
| 6 | Zero Inflation | Enters the ZI equation |
| 7 | Membership only | Drives latent-class probability only |
| 8 | Membership + Fixed | Drives class prob AND has class-specific outcome effect |

Run `get_help('roles')` for the full distribution reference.

In [ ]:
# ── Variables to include in the search ───────────────────────────────────────
# Edit this list to use your own predictors.
# All variables must be columns in df.
variables = [
    'AADT',      # Annual average daily traffic — primary exposure driver
    'LENGTH',    # Segment length (miles)
    'SPEED',     # Posted speed limit (mph)
    'CURVES',    # Presence/density of horizontal curves
    'URB',       # Urban indicator (binary)
    'ACCESS',    # Access control type
    'GRADEBR',   # Grade break indicator
    'SLOPE',     # Roadway slope
    'AVEPRE',    # Average annual precipitation
]

# ── Structural constraints ────────────────────────────────────────────────────
c = (
    ModelConstraints()

    # OFFSET must always be in the model (log-exposure; never exclude it)
    .force_include('OFFSET')

    # Geometric road features: implausible as zero-inflation terms
    .no_zi('LENGTH', 'CURVES', 'GRADEBR', 'WIDTH', 'SLOPE')

    # Binary indicators: no individual taste variation makes sense
    .no_random('URB', 'GRADEBR')

    # Allow curvature to be random — but positive-only (lognormal)
    .allow_random('CURVES', distributions=['lognormal'])

    # Precipitation can only explain variation in means (role 5), not be random itself
    # Remove this line if you want AVEPRE to be free to take any role
    # .set_roles('AVEPRE', [0, 5])
)

# Display the active constraints
print('Active constraints:')
print(c)

## 3 — Build the experiment and evaluator

In [ ]:
builder = ExperimentBuilder(
    df=df,
    id_col='ID',
    y_col='FREQ',
    offset_col='OFFSET',
    group_id_col='FC',     # panel grouping — pass None if not applicable
)

builder.describe()

In [ ]:
# ── What can you change here? ─────────────────────────────────────────────────
#
#   default_roles   : which roles ANY variable may take by default
#                     [0,1,2,3]      = exclude / fixed / random only
#                     [0,1,2,3,5]    = + heterogeneity in means
#                     [0,1,2,3,5,6]  = + zero inflation
#                     [0,1,2,3,5,7,8]= + latent-class membership (need LC>=2)
#
#   max_latent_classes : 1 = standard model, 2+ = latent class model
#
#   R               : Halton simulation draws
#                     200  = fast search (use for exploration)
#                     500  = stable estimation
#                     1000 = high accuracy (slow)
#
#   mode            : 'single' = minimise BIC  |  'multi' = [BIC, RMSE]

evaluator = builder.build_evaluator(
    variables=variables,
    constraints=c,
    default_roles=[0, 1, 2, 3, 5],
    max_latent_classes=1,
    mode='single',
    R=200,
)

print('Evaluator ready.')
print('Search over', len(variables), 'variables with roles:', [0, 1, 2, 3, 5])

## 4 — Run the structure search

The search explores candidate model structures and keeps the one with the lowest BIC.

**Choosing `max_iter`:**
- `50–100` — quick sanity check / first look
- `500–1000` — moderate search (minutes)
- `2000–5000` — production search (tens of minutes)
- `10000+` — overnight / HPC run (set `max_time` from walltime)

**Choosing `algo`:**
- `'sa'` — Simulated Annealing (best default)
- `'de'` — Differential Evolution (thorough, slower)
- `'hs'` — Harmony Search (fast initial convergence)

In [ ]:
print('Running structure search ...')
print('(max_iter=50 for demo — increase to 1000+ for production)')
print()

result = builder.run(
    evaluator=evaluator,
    algo='sa',
    max_iter=50,
    seed=42,
    output_config=SearchOutputConfig(
        output_dir='results',
        experiment_name='nb_crash_freq_search',
        search_description='Mixed NB count model search on Example 16-3',
    ),
)

print('─' * 60)
print(f'  Algorithm        : {result.algorithm}')
print(f'  Best BIC         : {result.best_score:.3f}')
print(f'  Iterations run   : {result.iteration}')
print(f'  Time elapsed     : {result.elapsed_time:.1f} s')
print(f'  Result saved to  : {result.saved_to}')
print('─' * 60)

In [ ]:
# Inspect the best structure found
print('Best model structure:')
print()
best = result.best_spec
for key, val in best.items():
    if val:  # only print non-empty fields
        print(f'  {key:<20}: {val}')

## 5 — Re-fit the winning structure with more draws

The search uses `R=200` for speed.  Re-fit with `R=500` to get accurate
parameter estimates and standard errors.

In [ ]:
print('Re-fitting best structure with R=500 draws ...')
print()

fit = builder.fit_manual_model(
    manual_spec=result.best_spec,
    model='nb',    # 'nb' = Negative Binomial  |  'poisson' = Poisson
    R=500,
)

print('─' * 60)
print('Final fitted model:')
print('─' * 60)
print(fit)

## 6 — Fit a manually specified structure

Use `make_manual_spec()` to specify exactly which role and distribution
every variable takes — no search needed.

This is useful when you:
- Want to replicate a published model
- Have theoretical reasons to fix a structure
- Want to compare 2 specific structures head-to-head

In [ ]:
# ── What you can specify in make_manual_spec ──────────────────────────────────
#
#   fixed_terms       : list of variable names → role 1 (fixed)
#   rdm_terms         : list of 'VAR:dist' strings → role 2 (random, independent)
#   rdm_cor_terms     : list of 'VAR:dist' strings → role 3 (random, correlated)
#   grouped_terms     : list of 'VAR:dist' strings → role 4 (grouped random)
#   hetro_in_means    : list of variable names → role 5 (heterogeneity)
#   zi_terms          : list of variable names → role 6 (zero inflation)
#   membership_terms  : list of variable names → role 7 (membership only)
#   dispersion        : int  0 = no dispersion  |  1 = estimate NB dispersion
#   latent_classes    : int  1 = standard model  |  2+ = latent class
#
#   Distributions available: 'normal', 'lognormal', 'triangular', 'uniform'

manual_spec = builder.make_manual_spec(
    fixed_terms=['AADT', 'LENGTH', 'SPEED'],
    rdm_terms=[],
    rdm_cor_terms=['CURVES:lognormal', 'SLOPE:normal'],
    grouped_terms=[],
    hetro_in_means=['AVEPRE'],
    zi_terms=['ACCESS'],
    membership_terms=[],
    dispersion=1,
    latent_classes=1,
)

print('Manual specification:')
for key, val in manual_spec.items():
    print(f'  {key:<20}: {val}')
print()

manual_fit = builder.fit_manual_model(
    manual_spec=manual_spec,
    model='nb',
    R=300,
)
print('─' * 60)
print('Manual model fit result:')
print('─' * 60)
print(manual_fit)

## 7 — Tuning tips

| What to change | How | Expected effect |
| --- | --- | --- |
| More variables | Add to `variables` list | Wider search space, slower per-iter |
| Allow ZI | Add `6` to `default_roles` | Model can explain structural zeros |
| Allow latent classes | `max_latent_classes=2`, add `7,8` to `default_roles` | More flexible heterogeneity |
| More accuracy | Increase `R` (200→500→1000) | Slower but more stable |
| Longer search | Increase `max_iter` | Better BIC, more time |
| Different algorithm | `algo='de'` or `algo='hs'` | Different exploration strategy |

**Next:** see [02_latent_class_fc_validation.ipynb](02_latent_class_fc_validation.ipynb)
to fit a 2-class LC model and check whether the classes align with the known functional class.